# 03 · Transformer blocks and sequence objectives

**Transformer Models · Notebook 3 of 4**

## What you will build

Attention is one part of a transformer. In this notebook you build the rest of a block: **layer normalisation**, the
position-wise **MLP** and the **residual connections** that tie them into a pre-norm transformer block. Then you build
the training objective of a causal language model: inputs and **next-token targets** made by shifting once, and a loss
that counts only real target tokens, normalised by how many there are. You compare that normalisation with two common
alternatives, and you finish by checking the gradients of a whole block against numerical differentiation.

## Learning objectives

By the end of this notebook you will be able to:

1. implement layer normalisation over each position's features, and say why it is per position;
2. implement the position-wise MLP and a pre-norm block $x + \mathrm{Attn}(\mathrm{LN}(x))$, then $+\,\mathrm{MLP}(\mathrm{LN}(\cdot))$;
3. show by measurement what residual connections preserve in a deep stack;
4. build next-token inputs and targets, and a padding-masked loss normalised by the number of real tokens;
5. verify a whole block's gradients numerically.

## Dependencies

**Tier 1 (NumPy and Matplotlib); this notebook needs only NumPy.** Everything runs on NumPy and the tiny tensor
engine (hidden recap cell), and there are no plots. No internet access is needed; the notebook runs in JupyterLite, on
JupyterHub and in Colab in a few seconds.

### How to use this notebook

Each building block follows the same rhythm: a short explanation, a tiny demo, then **✍️ Your turn**, where you type a function into a stub and run the check cell under it. The check prints ✅ when your function matches the reference and ❌, with the failing input, when it does not. Until your version passes, the notebook quietly uses the reference, so *Run all* always works; once it passes, every later cell runs on **your** code. Cells marked 🧪 are experiments: change the marked setting and run them again.

## Setup

Run this cell first. It loads NumPy and defines `practice`, the checker used after every ✍️ cell.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, unchanged. `Tensor` supports `+ - * / ** @`, `exp`, `log`, `tanh`, `relu`, `sigmoid`, `sum`, `mean`, `reshape`, `transpose`, NumPy-style indexing, `softmax`, `log_softmax` and `backward()`; next to it are `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it has been checked against numerical gradients. Run the cell, and open it if you want to see again how an operation passes its gradient back.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

### One more checker: `practice_tensor`

`practice` compares plain values, but most functions in this course take and return `Tensor`s. `practice_tensor` (hidden below) runs your function and the reference on the same inputs and compares two things: the values they return, and the gradients that flow back to every input that requires one. A function that slips into raw NumPy (through `.data`, `np.exp` or `float()`, for example) can return the right numbers while cutting the gradient path, so training would silently stop updating some weights. The second comparison catches that.

In [ ]:
# ── practice_tensor: practice() for functions built from Tensor operations ──


def _tensors_in(value):
    """Every Tensor inside value, looking into tuples, lists and dicts."""
    if isinstance(value, Tensor):
        return [value]
    if isinstance(value, dict):
        value = list(value.values())
    if isinstance(value, (list, tuple)):
        return [tensor for item in value for tensor in _tensors_in(item)]
    return []


def _values_and_gradients(function, args):
    """Run function(*args). Return its output arrays and the gradients it sends back to each input Tensor."""
    inputs = [tensor for tensor in _tensors_in(args) if tensor.requires_grad]
    zero_grad(inputs)
    result = function(*args)
    outputs = list(result) if isinstance(result, tuple) else [result]
    for output in outputs:
        if not isinstance(output, Tensor):
            hint = ("; a NumPy array on the left of + or @ turns the result into an object array, so put the Tensor first"
                    if getattr(output, "dtype", None) == object else "; build the result from Tensor operations")
            raise TypeError(f"expected a Tensor but got {type(output).__name__}{hint}")
    probe = Tensor(0.0)
    for output in outputs:  # an uneven weighting, so that no gradient cancels out by symmetry
        probe = probe + (output * np.cos(np.arange(output.data.size)).reshape(output.shape)).sum()
    probe.backward()
    values = [output.data.copy() for output in outputs]
    gradients = [np.zeros_like(tensor.data) if tensor.grad is None else tensor.grad.copy() for tensor in inputs]
    zero_grad(inputs)
    return values, gradients


def practice_tensor(name, yours, reference, cases, atol=1e-6):
    """Like practice(), for functions built from Tensor operations.

    It checks the values your function returns AND the gradients that flow back
    through them, so code that slips into raw NumPy (which cuts the gradient
    path) does not pass. Returns yours once it matches, otherwise the reference.
    """
    try:
        for args in cases:
            got, got_gradients = _values_and_gradients(yours, args)
            want, want_gradients = _values_and_gradients(reference, args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got[0] if len(got) == 1 else tuple(got))}")
                print(f"   expected     {_short(want[0] if len(want) == 1 else tuple(want))}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
            if not _close(got_gradients, want_gradients, atol):
                print(f"❌ {name}: the values match, but the gradients flowing back through your function do not.")
                print("   Use Tensor operations only: .data, np.* functions and float() cut the gradient path.")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: values and gradients match the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. "
          f"From here on the notebook runs YOUR {name}.")
    return yours

## 📦 Recap: what you built earlier

The hidden cell defines your pieces from notebooks 01 and 02 under their usual names: `encode`, `decode`,
`embedding_lookup`, `sinusoidal_positions`, `swap_last_two`, `attention`, `causal_mask`, `padding_mask`,
`split_heads`, `merge_heads`, `init_attention` and `multi_head_attention`.

In [ ]:
# 📦 Recap: the pieces from notebooks 01 and 02, under their usual names.


def encode(text, stoi):
    """Return the list of token ids for the characters of text, in order."""
    return [stoi[character] for character in text]


def decode(ids, itos):
    """Return the string spelled by a sequence of token ids."""
    return "".join(itos[index] for index in ids)


def embedding_lookup(table, ids):
    """Rows of the embedding table for each id. table: Tensor (vocab, width); ids: ints of any shape."""
    return table[ids]


def sinusoidal_positions(length, width):
    """Sinusoidal position signals, shape (length, width); width must be even."""
    positions = np.arange(length)[:, None]
    rates = 1.0 / 10000 ** (np.arange(0, width, 2) / width)
    signals = np.zeros((length, width))
    signals[:, 0::2] = np.sin(positions * rates)
    signals[:, 1::2] = np.cos(positions * rates)
    return signals


def swap_last_two(t):
    """(..., L, d) -> (..., d, L). The engine's transpose takes the full order of axes."""
    axes = list(range(t.ndim))
    axes[-2], axes[-1] = axes[-1], axes[-2]
    return t.transpose(*axes)


def attention(q, k, v, mask=None):
    """Scaled dot-product attention. Returns (output, weights)."""
    scores = q @ swap_last_two(k) / np.sqrt(q.shape[-1])
    if mask is not None:
        scores = scores + np.where(mask, 0.0, -1e9)
    weights = scores.softmax(axis=-1)
    return weights @ v, weights


def causal_mask(length):
    """Boolean (length, length): True where key position <= query position."""
    return np.tril(np.ones((length, length), dtype=bool))


def padding_mask(token_ids, pad_id):
    """Boolean (batch, 1, length): True where the key is a real token, not padding."""
    return (np.asarray(token_ids) != pad_id)[:, None, :]


def split_heads(x, n_heads):
    """(batch, length, width) -> (batch, heads, length, width // heads)."""
    batch, length, width = x.shape
    return x.reshape(batch, length, n_heads, width // n_heads).transpose(0, 2, 1, 3)


def merge_heads(x):
    """(batch, heads, length, head_width) -> (batch, length, heads * head_width)."""
    batch, heads, length, head_width = x.shape
    return x.transpose(0, 2, 1, 3).reshape(batch, length, heads * head_width)


def init_attention(width, rng):
    """Query, key, value and output projections, each (width, width)."""
    return {name: parameter(width, width, rng=rng, scale=width ** -0.5) for name in ("Wq", "Wk", "Wv", "Wo")}


def multi_head_attention(x, params, n_heads, mask=None):
    """Self-attention with n_heads heads. x: (B, L, D). mask: boolean, broadcastable to (B, L, L).
    Returns (output (B, L, D), weights (B, heads, L, L))."""
    q = split_heads(x @ params["Wq"], n_heads)
    k = split_heads(x @ params["Wk"], n_heads)
    v = split_heads(x @ params["Wv"], n_heads)
    if mask is not None:
        mask = np.asarray(mask)[..., None, :, :]  # add a head axis: (L, L) -> (1, L, L); (B, L, L) -> (B, 1, L, L)
    out, weights = attention(q, k, v, mask)
    return merge_heads(out) @ params["Wo"], weights

## 1 · Layer normalisation

Deep stacks of layers drift: activations can grow or shrink from block to block, which makes training fragile. **Layer
normalisation** (Ba, Kiros and Hinton, 2016) rescales each position's feature vector $x \in \mathbb{R}^D$ to mean 0
and variance 1, then applies a learned scale $\gamma$ and shift $\beta$:

$$\mu = \frac{1}{D}\sum_{c} x_c, \qquad \sigma^2 = \frac{1}{D}\sum_{c} (x_c - \mu)^2, \qquad \mathrm{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta .$$

The statistics are taken over the **last axis only**, separately for every position of every sequence. Unlike batch
normalisation, nothing is shared across the batch, so a model behaves the same with one sequence (during generation) as
with sixty-four. **Shapes:** $x$ is $(B, L, D)$; $\gamma$ and $\beta$ are $(D,)$; the output is $(B, L, D)$.

The engine recap already contains a `layer_norm`. Here you write it yourself from the formula, and your version
replaces it.

In [ ]:
x_demo = np.array([1.0, 2.0, 3.0, 6.0])
mean, variance = x_demo.mean(), ((x_demo - x_demo.mean()) ** 2).mean()
print("mean", mean, "| variance", variance, "| normalised", ((x_demo - mean) / np.sqrt(variance + 1e-5)).round(3))

### ✍️ Your turn: `layer_norm`

Write `layer_norm(x, gamma, beta, eps=1e-5)` for a `Tensor` `x` of shape `(..., D)` and `Tensor`s `gamma` and `beta`
of shape `(D,)`. Follow the formula above, taking every mean over the last axis.

*Hints:* `x.mean(axis=-1, keepdims=True)` keeps a trailing axis of length 1, so the result broadcasts back against `x`.
For the square root, use `** 0.5` on the `Tensor`; `np.sqrt` would cut the gradient path.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def layer_norm_reference(x, gamma, beta, eps=1e-5):
    """Normalise each position's feature vector (the last axis), then scale and shift."""
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    variance = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (variance + eps) ** 0.5 * gamma + beta

In [ ]:
def layer_norm(x, gamma, beta, eps=1e-5):
    """Normalise each position's feature vector (the last axis), then scale and shift."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
rng_check = np.random.default_rng(3)
gamma_check = Tensor(rng_check.normal(size=4), requires_grad=True)
beta_check = Tensor(rng_check.normal(size=4), requires_grad=True)
flat_row = rng_check.normal(size=(2, 3, 4))
flat_row[0, 1] = 5.0  # a position whose features are all equal: its variance is 0, so eps matters
layer_norm = practice_tensor("layer_norm", layer_norm, layer_norm_reference, cases=[
    (Tensor(rng_check.normal(size=(2, 3, 4)), requires_grad=True), gamma_check, beta_check),
    (Tensor(rng_check.normal(size=(1, 1, 4)), requires_grad=True), gamma_check, beta_check),
    (Tensor(flat_row, requires_grad=True), gamma_check, beta_check),
])

**Expected behaviour.** With $\gamma = 1$ and $\beta = 0$, every position comes out with mean 0 and standard deviation
almost exactly 1, whatever its original scale. A position whose features are all equal comes out as $\beta$, and
`eps` keeps that case from dividing by zero.

In [ ]:
x_scales = Tensor(np.random.default_rng(4).normal(size=(2, 3, 8)) * np.array([[[0.1]], [[50.0]]]))
normalised = layer_norm(x_scales, Tensor(np.ones(8)), Tensor(np.zeros(8))).data
print("means:", normalised.mean(axis=-1).round(6).ravel())
print("standard deviations:", normalised.std(axis=-1).round(4).ravel())

## 2 · The position-wise MLP

Attention moves information *between* positions; the **MLP** (also called the feed-forward network) then transforms
each position *on its own*:

$$\mathrm{MLP}(x) = \mathrm{ReLU}(x W_1 + b_1)\, W_2 + b_2, \qquad W_1: (D, F),\ b_1: (F,),\ W_2: (F, D),\ b_2: (D,).$$

The same weights apply at every position, and the hidden width $F$ is usually about $4D$, so most of a transformer's
parameters live here. GPT-style models often use GELU instead of ReLU (exercise 2). **Shapes:**
$(B, L, D) \to (B, L, F) \to (B, L, D)$.

In [ ]:
D_demo, F_demo = 32, 128
print("MLP parameters at D = 32, F = 128:", D_demo * F_demo + F_demo + F_demo * D_demo + D_demo)
print("attention projections at D = 32:", 4 * D_demo * D_demo)

### ✍️ Your turn: `mlp`

Write `mlp(x, params)` for a `Tensor` `x` of shape `(..., D)`, where `params` is a dict holding the `Tensor`s `"W1"`,
`"b1"`, `"W2"` and `"b2"`. Return $\mathrm{ReLU}(xW_1 + b_1)W_2 + b_2$.

*Hint:* the engine's `Tensor` has a `.relu()` method. One line.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def mlp_reference(x, params):
    """Position-wise feed-forward network: ReLU(x W1 + b1) W2 + b2."""
    return (x @ params["W1"] + params["b1"]).relu() @ params["W2"] + params["b2"]

In [ ]:
def mlp(x, params):
    """Position-wise feed-forward network: ReLU(x W1 + b1) W2 + b2."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
mlp_check = {"W1": parameter(4, 6, rng=rng_check), "b1": Tensor(rng_check.normal(size=6), requires_grad=True),
             "W2": parameter(6, 4, rng=rng_check), "b2": Tensor(rng_check.normal(size=4), requires_grad=True)}
mlp = practice_tensor("mlp", mlp, mlp_reference, cases=[
    (Tensor(rng_check.normal(size=(2, 3, 4)), requires_grad=True), mlp_check),
    (Tensor(rng_check.normal(size=(1, 1, 4)), requires_grad=True), mlp_check),
])

**Expected behaviour.** Because the MLP treats positions independently, running it on a whole `(B, L, D)` batch gives
the same rows as running it on each position separately. The next cell checks this.

In [ ]:
x_batch = Tensor(rng_check.normal(size=(2, 3, 4)))
whole = mlp(x_batch, mlp_check).data
one_by_one = np.stack([[mlp(Tensor(x_batch.data[b, i]), mlp_check).data for i in range(3)] for b in range(2)])
print("whole batch equals position by position:", np.allclose(whole, one_by_one))

## 3 · Residual connections and the pre-norm block

A **residual connection** adds a sublayer's output to its input: $y = x + f(x)$. The sublayer learns a *change* to
$x$ rather than a replacement, and the gradient has a direct path back, $\partial y / \partial x = I + \partial f /
\partial x$, even when $\partial f/\partial x$ is small (He et al., 2016). Dimensions must agree, which is why every
sublayer maps width $D$ back to width $D$.

A **pre-norm** transformer block normalises the input of each sublayer and adds the result back:

$$h = x + \mathrm{MHA}(\mathrm{LN}_1(x)), \qquad \mathrm{out} = h + \mathrm{MLP}(\mathrm{LN}_2(h)).$$

The original transformer used **post-norm**, $\mathrm{LN}(x + f(x))$ (exercise 1). Pre-norm keeps an untouched residual
stream from input to output and tends to train more stably in deep stacks (Xiong et al., 2020); GPT-2-style models use
it.

In [ ]:
x_res = Tensor(np.array([1.0, -2.0, 0.5]), requires_grad=True)
y_res = x_res + (x_res * 0.0).tanh()  # a sublayer whose own gradient is zero
y_res.sum().backward()
print("gradient through the residual path:", x_res.grad)

`init_block` (given) creates one block's parameters: the four attention projections, two layer norms
($\gamma = 1$, $\beta = 0$) and the MLP. The output projections $W_O$ and $W_2$ start smaller, so a fresh block adds only
a modest change to its input.

In [ ]:
def init_block(width, hidden, rng, n_blocks=1):
    """Parameters of one pre-norm block. Output projections start small so that a deep stack starts calm."""
    params = init_attention(width, rng)
    params["Wo"].data *= (2 * n_blocks) ** -0.5
    params.update({
        "ln1_g": Tensor(np.ones(width), requires_grad=True), "ln1_b": zeros_parameter(width),
        "ln2_g": Tensor(np.ones(width), requires_grad=True), "ln2_b": zeros_parameter(width),
        "W1": parameter(width, hidden, rng=rng), "b1": zeros_parameter(hidden),
        "W2": parameter(hidden, width, rng=rng, scale=(hidden * 2 * n_blocks) ** -0.5), "b2": zeros_parameter(width),
    })
    return params


block_demo = init_block(8, 16, np.random.default_rng(5))
print({name: tensor.shape for name, tensor in block_demo.items()})

### ✍️ Your turn: `transformer_block`

Write `transformer_block(x, params, n_heads, mask=None)` for a `Tensor` `x` of shape `(B, L, D)` and a parameter dict
from `init_block`. Implement the pre-norm block:

1. `attended, _ = multi_head_attention(layer_norm(x, params["ln1_g"], params["ln1_b"]), params, n_heads, mask)`;
2. `x = x + attended`;
3. return `x + mlp(layer_norm(x, params["ln2_g"], params["ln2_b"]), params)`.

*Hint:* `multi_head_attention` reads `Wq`, `Wk`, `Wv` and `Wo` from the same dict, and `mlp` reads `W1`, `b1`, `W2` and
`b2`, so you can pass `params` to both.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def transformer_block_reference(x, params, n_heads, mask=None):
    """Pre-norm block: x + attention(LN1(x)), then + MLP(LN2(.))."""
    attended, _ = multi_head_attention(layer_norm(x, params["ln1_g"], params["ln1_b"]), params, n_heads, mask)
    x = x + attended
    return x + mlp(layer_norm(x, params["ln2_g"], params["ln2_b"]), params)

In [ ]:
def transformer_block(x, params, n_heads, mask=None):
    """Pre-norm block: x + attention(LN1(x)), then + MLP(LN2(.))."""
    # ✍️ Your code here (three lines)
    raise NotImplementedError

In [ ]:
block_check = init_block(8, 16, rng_check)
block_check["b1"].data += rng_check.normal(size=16)  # non-zero biases, so that a forgotten term shows up
transformer_block = practice_tensor("transformer_block", transformer_block, transformer_block_reference, cases=[
    (Tensor(rng_check.normal(size=(2, 4, 8)), requires_grad=True), block_check, 2, causal_mask(4)),
    (Tensor(rng_check.normal(size=(1, 3, 8)), requires_grad=True), block_check, 4),
])

**Expected behaviour.** The output has the same shape as the input, so blocks can be stacked. The check compares the
values and the gradients of all twelve parameters and of `x`.

## 4 · Next-token targets

A causal language model learns $p(t_{i+1} \mid t_0, \dots, t_i)$ at every position at once. For a sequence
$[t_0, t_1, \dots, t_n]$ the inputs are $[t_0, \dots, t_{n-1}]$ and the targets are $[t_1, \dots, t_n]$: position $i$
predicts token $i + 1$, and the causal mask from notebook 02 stops it from seeing that token. Shift **exactly once**.
Some library model classes shift internally, while a hand-written loss expects you to do it; shifting twice, or not at
all, still produces a loss, just for the wrong task.

In this course every sentence starts and ends with a newline, `"\n"`, which marks the start for the first prediction
and the end of the sentence for the last one. With padding, the targets of padded positions are `<pad>` (id 0), and
those positions must not be scored: the **loss mask** is `targets != 0`.

In [ ]:
sentences = ["the cat likes the dog.", "the birds call the frogs.", "the hen helps the cats."]
itos = ["<pad>"] + sorted(set("\n".join(sentences)) | {"\n"})
stoi = {token: index for index, token in enumerate(itos)}
row = encode("\nthe cat.\n", stoi)
for current, following in zip(row[:-1], row[1:]):
    print(f"input {itos[current]!r:6} -> target {itos[following]!r}")

### ✍️ Your turn: `next_token_pairs`

Write `next_token_pairs(token_ids)` for an integer array of shape `(B, L)`. Return the tuple `(inputs, targets)`, each
of shape `(B, L - 1)`: the inputs drop the last column and the targets drop the first.

*Hint:* slicing on the second axis.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def next_token_pairs_reference(token_ids):
    """Inputs and targets for next-token prediction: shift by exactly one position."""
    token_ids = np.asarray(token_ids)
    return token_ids[:, :-1], token_ids[:, 1:]

In [ ]:
def next_token_pairs(token_ids):
    """Inputs and targets for next-token prediction: shift by exactly one position."""
    # ✍️ Your code here (about two lines)
    raise NotImplementedError

In [ ]:
next_token_pairs = practice("next_token_pairs", next_token_pairs, next_token_pairs_reference, cases=[
    (np.array([[1, 5, 6, 7, 2]]),),
    (np.array([[1, 5, 2, 0], [1, 6, 7, 2]]),),
    (np.array([[3, 4]]),),
])

## 5 · A loss that counts only real tokens

For logits $z_{b,i} \in \mathbb{R}^V$ and target $y_{b,i}$, the negative log-likelihood at one position is
$\ell_{b,i} = -\log \mathrm{softmax}(z_{b,i})_{y_{b,i}}$. With a mask $m_{b,i} \in \{0, 1\}$ that is 1 for real targets,
the loss is the mean over **real target tokens**:

$$\mathcal{L} = \frac{\sum_{b,i} m_{b,i}\, \ell_{b,i}}{\sum_{b,i} m_{b,i}} .$$

This matches what frameworks compute with an ignore index and mean reduction (in PyTorch, `cross_entropy(...,
ignore_index=pad)`), and it equals the engine's `cross_entropy(logits, targets, mask)`. Two other averages look
reasonable but behave differently, and section 6 measures them:

- dividing by **all** $B \cdot L$ positions makes the loss shrink when a batch contains more padding;
- averaging **each sequence first** and then the sequences gives every token of a short sentence more weight than a
  token of a long one.

A batch with no real targets has a denominator of 0. Detect it before you call the loss: the training code in notebook
04 asserts that every batch has real targets.

**Shapes:** logits $(B, L, V)$; targets and mask $(B, L)$; the loss is a scalar `Tensor`.

In [ ]:
logits_demo = Tensor(np.array([[2.0, 0.5, -1.0], [0.0, 0.0, 3.0]]))  # 2 positions, vocabulary of 3
log_probs = logits_demo.log_softmax(axis=-1)
targets_demo = np.array([0, 1])
print("log-probabilities of the targets:", log_probs.data[np.arange(2), targets_demo].round(3))

### ✍️ Your turn: `masked_token_loss`

Write `masked_token_loss(logits, targets, mask)`. `logits` is a `Tensor` of shape `(B, L, V)`, and `targets` (ints)
and `mask` (booleans or 0/1) are NumPy arrays of shape `(B, L)`. Return the scalar `Tensor` $\mathcal{L}$ defined above.
You may assume that the mask contains at least one real token.

*Hints:* take `logits.log_softmax(axis=-1)`, flatten it with `.reshape(-1, V)`, and pick each target's log-probability
with `[np.arange(n), targets.reshape(-1)]`. Multiply by the flattened mask as floats, `.sum()`, then divide by the
number of real tokens.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def masked_token_loss_reference(logits, targets, mask):
    """Mean negative log-likelihood over real target tokens. logits: (B, L, V); targets, mask: (B, L)."""
    log_probs = logits.log_softmax(axis=-1).reshape(-1, logits.shape[-1])
    targets = np.asarray(targets).reshape(-1)
    picked = log_probs[np.arange(targets.size), targets]
    weights = np.asarray(mask, dtype=float).reshape(-1)
    return -(picked * weights).sum() / weights.sum()

In [ ]:
def masked_token_loss(logits, targets, mask):
    """Mean negative log-likelihood over real target tokens. logits: (B, L, V); targets, mask: (B, L)."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
logits_check = Tensor(rng_check.normal(size=(2, 3, 5)), requires_grad=True)
targets_check = np.array([[1, 4, 0], [2, 0, 0]])
masked_token_loss = practice_tensor("masked_token_loss", masked_token_loss, masked_token_loss_reference, cases=[
    (logits_check, targets_check, targets_check != 0),
    (logits_check, targets_check, np.ones((2, 3))),
    (Tensor(rng_check.normal(size=(1, 1, 4)), requires_grad=True), np.array([[3]]), np.array([[1.0]])),
])
print("matches the engine's cross_entropy:",
      np.isclose(masked_token_loss(logits_check, targets_check, targets_check != 0).data,
                 cross_entropy(logits_check, targets_check, targets_check != 0).data))

## 6 · 🧪 Experiments

### What residual connections preserve

Stack `depth` blocks with fresh random weights and feed in 12 random position vectors, using bidirectional attention.
At initialisation, attention weights are spread fairly evenly, so each block's attention output is close to an
*average* of the positions. Without residual connections the next block only sees those averages, and the positions quickly become
indistinguishable. With residual connections each position keeps its own vector and only adds changes. The table
measures the average cosine similarity between different positions after the stack: 1 means that all positions carry the
same vector.

In [ ]:
def block_without_residuals(x, params, n_heads, mask=None):
    """The same sublayers as transformer_block, with both residual connections removed."""
    attended, _ = multi_head_attention(layer_norm(x, params["ln1_g"], params["ln1_b"]), params, n_heads, mask)
    return mlp(layer_norm(attended, params["ln2_g"], params["ln2_b"]), params)


def mean_cosine_similarity(h):
    vectors = h.data[0] / np.linalg.norm(h.data[0], axis=-1, keepdims=True)
    similarity = vectors @ vectors.T
    n = len(vectors)
    return (similarity.sum() - n) / (n * (n - 1))


depths = (1, 4, 16)  # 🧪 add 32 and run again
print(" depth   similarity with residuals   similarity without")
for depth in depths:
    row = []
    for block_function in (transformer_block, block_without_residuals):
        rng_depth = np.random.default_rng(0)
        blocks = [init_block(16, 32, rng_depth, n_blocks=depth) for _ in range(depth)]
        h = Tensor(rng_depth.normal(size=(1, 12, 16)))
        for params in blocks:
            h = block_function(h, params, 4)
        row.append(mean_cosine_similarity(h))
    print(f"{depth:6d}   {row[0]:25.3f}   {row[1]:18.3f}")

**What to notice.** With residual connections the positions stay distinct at every depth. Without them, a few blocks
are enough for every position to carry essentially the same vector: the information about *which* token sat where has
been averaged away before training even starts. Dong, Cordonnier and Loukas (2021) analyse this collapse in pure
attention networks.

### Three ways to average a padded loss

One batch holds a short sentence (2 real targets) and a long one (8 real targets). The cell computes the token-level
mean (your `masked_token_loss`), the mean over every position, and the mean of per-sentence means. Then it pads the
same batch with `extra_padding` more columns and computes all three again.

In [ ]:
rng_loss = np.random.default_rng(1)
V_demo = 6
logits_pad = Tensor(rng_loss.normal(size=(2, 8, V_demo)), requires_grad=True)
targets_pad = rng_loss.integers(1, V_demo, size=(2, 8))
targets_pad[0, 2:] = 0  # the first sentence has only 2 real targets
extra_padding = 10  # 🧪 try 0 and 40


def three_averages(logits, targets):
    mask = targets != 0
    nll = -logits.log_softmax(axis=-1).data[np.arange(2)[:, None], np.arange(targets.shape[1])[None, :], targets]
    token_mean = float(masked_token_loss(logits, targets, mask).data)
    every_position = (nll * mask).sum() / mask.size
    per_sequence = ((nll * mask).sum(axis=1) / mask.sum(axis=1)).mean()
    return token_mean, every_position, per_sequence


wider_logits = Tensor(np.concatenate([logits_pad.data, rng_loss.normal(size=(2, extra_padding, V_demo))], axis=1))
wider_targets = np.concatenate([targets_pad, np.zeros((2, extra_padding), dtype=int)], axis=1)
print("                     token mean   every position   per-sequence mean")
print("batch as it is      %10.4f   %14.4f   %17.4f" % three_averages(logits_pad, targets_pad))
print("with extra padding  %10.4f   %14.4f   %17.4f" % three_averages(wider_logits, wider_targets))

**What to notice.** The token mean and the per-sequence mean ignore extra padding; the every-position mean shrinks as
padding is added, although the model has not changed. The per-sequence mean differs from the token mean because each of
the short sentence's 2 tokens counts four times as much as each of the long sentence's 8. When batches contain different
numbers of real tokens (always, with padding), reporting the token mean is what makes losses and perplexities
comparable.

## 7 · Put it together: one block, a loss and a gradient check

First, the full objective on a padded batch: embeddings and positions, one masked transformer block, an output
projection to the vocabulary and your loss. The asserts check the behaviour you will rely on in notebook 04:

1. inputs and targets are aligned by exactly one position;
2. a fresh model's loss is close to $\log V$, the loss of a uniform guess;
3. changing the tokens at padded positions, in the inputs or in the targets, leaves the loss exactly unchanged;
4. the gradients of the whole block (all twelve parameters and the input) agree with central differences.

In [ ]:
D, H, V = 16, 4, len(itos)
rng_model = np.random.default_rng(6)
table = parameter(V, D, rng=rng_model, scale=1.0)
block = init_block(D, 32, rng_model)
W_out = parameter(D, V, rng=rng_model, scale=0.01)  # small output weights: a fresh model predicts close to uniform

encoded = [encode("\n" + sentence + "\n", stoi) for sentence in sentences]
rows = np.zeros((len(encoded), max(map(len, encoded))), dtype=int)
for r, ids in enumerate(encoded):
    rows[r, :len(ids)] = ids
inputs, targets = next_token_pairs(rows)


def batch_loss(inputs, targets):
    L = inputs.shape[1]
    x = embedding_lookup(table, inputs) + sinusoidal_positions(L, D)
    h = transformer_block(x, block, H, causal_mask(L) & padding_mask(inputs, 0))
    return masked_token_loss(h @ W_out, targets, targets != 0)


loss = batch_loss(inputs, targets)
print(f"rows {rows.shape} -> inputs {inputs.shape}, targets {targets.shape}; real targets: {(targets != 0).sum()}")
print(f"loss of the fresh model {float(loss.data):.4f}, log V = {np.log(V):.4f}")

# Scramble every padded position: random real tokens in the inputs, random targets that the mask must ignore.
scrambled_inputs, scrambled_targets = inputs.copy(), targets.copy()
padded_inputs, padded_targets = inputs == 0, targets == 0
scrambled_inputs[padded_inputs] = rng_model.integers(1, V, size=padded_inputs.sum())
scrambled_targets[padded_targets] = rng_model.integers(1, V, size=padded_targets.sum())
scrambled_loss = batch_loss(scrambled_inputs, np.where(padded_targets, 0, scrambled_targets))
target_only_change = masked_token_loss(
    transformer_block(embedding_lookup(table, inputs) + sinusoidal_positions(inputs.shape[1], D), block, H,
                      causal_mask(inputs.shape[1]) & padding_mask(inputs, 0)) @ W_out,
    scrambled_targets, targets != 0)
print("loss with scrambled padded inputs:", float(scrambled_loss.data))
print("loss with scrambled padded targets (mask kept):", float(target_only_change.data))

In [ ]:
# The whole-block gradient check: width 8, 2 heads, causal and padding masks, both layer norms.
rng_grad = np.random.default_rng(7)
grad_block = init_block(8, 16, rng_grad)
for name in ("b1", "b2", "ln1_b", "ln2_b"):
    grad_block[name].data += 0.1 * rng_grad.normal(size=grad_block[name].shape)
for name in ("ln1_g", "ln2_g"):
    grad_block[name].data += 0.1 * rng_grad.normal(size=grad_block[name].shape)
x_grad = Tensor(rng_grad.normal(size=(2, 4, 8)), requires_grad=True)
grad_mask = causal_mask(4) & padding_mask(np.array([[3, 1, 2, 0], [1, 1, 2, 3]]), 0)
probe = rng_grad.normal(size=(2, 4, 8))


def block_objective():
    return (transformer_block(x_grad, grad_block, 2, grad_mask) * probe).sum()


block_objective().backward()
checked = list(grad_block.items()) + [("x", x_grad)]
errors = {name: relative_error(tensor.grad, numerical_grad(block_objective, tensor)) for name, tensor in checked}
worst = max(errors, key=errors.get)
print(f"checked {sum(t.data.size for _, t in checked)} numbers; worst relative error {errors[worst]:.1e} ({worst})")

In [ ]:
assert np.array_equal(inputs[:, 1:], targets[:, :-1]), "targets must be the inputs shifted by exactly one"
assert abs(float(loss.data) - np.log(V)) < 0.2, "a fresh model with small output weights should predict close to uniform"
assert abs(float(scrambled_loss.data) - float(loss.data)) < 1e-12, "padded inputs must not change the loss"
assert abs(float(target_only_change.data) - float(loss.data)) < 1e-12, "targets at padded positions must not be scored"
assert errors[worst] < 1e-6, f"gradient check failed for {worst}: relative error {errors[worst]:.1e}"
print("✅ Shift, loss mask and gradients verified: the block and the objective are ready for training in notebook 04.")

## Exercises

### ✍️ Your turn: `post_norm_block`

**Exercise 1.** Write the **post-norm** block of the original 2017 transformer, `post_norm_block(x, params, n_heads,
mask=None)`: `x = layer_norm(x + attended, ln1)`, where `attended` comes from `multi_head_attention(x, ...)` on the
*un-normalised* `x`, then return `layer_norm(x + mlp(x, params), ln2)`. Afterwards, run the residual experiment of
section 6 with it. What happens to the residual stream?

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def post_norm_block_reference(x, params, n_heads, mask=None):
    """Post-norm block of the 2017 paper: LN1(x + attention(x)), then LN2(. + MLP(.))."""
    attended, _ = multi_head_attention(x, params, n_heads, mask)
    x = layer_norm(x + attended, params["ln1_g"], params["ln1_b"])
    return layer_norm(x + mlp(x, params), params["ln2_g"], params["ln2_b"])

In [ ]:
def post_norm_block(x, params, n_heads, mask=None):
    """Post-norm block of the 2017 paper: LN1(x + attention(x)), then LN2(. + MLP(.))."""
    # ✍️ Your code here (three lines)
    raise NotImplementedError

In [ ]:
post_norm_block = practice_tensor("post_norm_block", post_norm_block, post_norm_block_reference, cases=[
    (Tensor(rng_check.normal(size=(2, 4, 8)), requires_grad=True), block_check, 2, causal_mask(4)),
])

### ✍️ Your turn: `gelu`

**Exercise 2.** Many GPT-style models replace ReLU with **GELU**. Write its common tanh approximation for a `Tensor`
`x`: $\mathrm{GELU}(x) \approx 0.5\,x\,\bigl(1 + \tanh\bigl(\sqrt{2/\pi}\,(x + 0.044715\,x^3)\bigr)\bigr)$. Use only
`Tensor` operations, so that `practice_tensor`'s gradient comparison passes.

Type it rather than pasting it: typing is how the shapes and indices sink in. The reference is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def gelu_reference(x):
    """GELU, tanh approximation: 0.5 x (1 + tanh(sqrt(2 / pi) (x + 0.044715 x^3)))."""
    return 0.5 * x * ((np.sqrt(2.0 / np.pi) * (x + 0.044715 * x ** 3)).tanh() + 1.0)

In [ ]:
def gelu(x):
    """GELU, tanh approximation: 0.5 x (1 + tanh(sqrt(2 / pi) (x + 0.044715 x^3)))."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
gelu = practice_tensor("gelu", gelu, gelu_reference, cases=[
    (Tensor(np.linspace(-4, 4, 9), requires_grad=True),),
    (Tensor(rng_check.normal(size=(2, 3)), requires_grad=True),),
])

**Exercise 3 (question).** *Weight tying* reuses the embedding table as the output projection: logits $= hE^\top$. How
many parameters does it save here ($V = 21$, $D = 16$), and for a vocabulary of 50,257 tokens at width 768? Which shapes
must match for it to work?

**Exercise 4 (question).** Why should a fresh model's loss be close to $\log V$? If the first loss you see is 30, what
would you check first?

## Common errors and debugging

- **Normalising over the wrong axis.** `x.mean(axis=0)` or `axis=1` mixes statistics across sequences or positions,
  and the model then behaves differently with different batch contents. Layer norm uses `axis=-1`.
- **Forgetting `keepdims=True`.** The mean of shape `(B, L)` cannot broadcast against `(B, L, D)`.
- **`np.sqrt` on a `Tensor`.** It returns raw NumPy and cuts the gradient path; use `** 0.5`.
- **A residual that does not match.** `x + f(x)` needs `f` to return width $D$. A shape error here usually means a
  projection has its axes swapped.
- **Shifting twice, or not at all.** Shifting twice trains a model to predict two tokens ahead. Not shifting trains it
  to copy its own input, so the loss falls to almost 0 while generation is useless.
- **Scoring padded targets, or dividing by every position.** Both make the loss depend on how much padding a batch
  contains.
- **A batch with no real targets.** The token mean becomes 0/0 = `nan`, and `nan` spreads into every weight at the next
  update. Check the count first.

## Summary

- Layer normalisation rescales each position's features to mean 0 and variance 1, then applies $\gamma$ and $\beta$.
- The MLP transforms each position independently and holds most of the parameters.
- Residual connections keep each position's own information and give gradients a direct path; a pre-norm block is
  $x + \mathrm{MHA}(\mathrm{LN}(x))$, then $+\,\mathrm{MLP}(\mathrm{LN}(\cdot))$.
- Next-token targets are the inputs shifted by one. The loss is the mean over real target tokens, and the attention
  mask and the loss mask are separate decisions.
- A numerical gradient check confirms that the whole block, masks and normalisation included, differentiates correctly.

## Next

Notebook 04 stacks these pieces into a complete character model, trains it on a grammar with a long-range rule, and
then generates text from it with greedy, temperature and top-k decoding.